# Calculate centered dipole magnetic latitude/longitude (IGRF-14)

This notebook:
- reads station geographic lat/lon from each event's site CSV
- computes centered dipole magnetic coordinates using IGRF-14 **at each event's own epoch**
- writes cdm_lat / cdm_lon back to CSV for MATLAB analysis
- computes the same coordinates for the map landmark cities (config/event_landmarks_geo_info.csv)

Convention locked: big-picture CDM coordinates (not fine sub-degree precision).
`cdm_lat` is stored **negative** for northern sites (dipole axis points to the south geomagnetic pole);
MATLAB flips the sign (`-cdm_lat`).


### Set up

import critical libraries:

In [ ]:
import numpy as np
import pandas as pd
import ppigrf
import ppigrf.ppigrf as core

user inputs:

In [ ]:
from storm_tools import config
dpath = str(config.load_paths()['high_frequency_path']) + '/'
LAT_COL = "lat"
LON_COL = "lon"

# One entry per event: storm date (sets the IGRF epoch) + file prefix.
# Epoch = decimal year of the storm date (e.g. 1989-03-13 -> 1989.19).
EVENTS = {
    "1989-03-13": "198903",
    "2003-10-29": "200310",
    "2024-05-11": "202405",
}
in_csv_fmt   = "{prefix}_event_sites_geo_info.csv"
out_csv_fmt  = "{prefix}_event_sites_geo_info_with_cdm.csv"

# Map landmark cities (tracked in the repo, read by Map_Events.m)
landmarks_in  = "config/event_landmarks_geo_info.csv"
landmarks_out = "config/event_landmarks_geo_info_with_cdm.csv"

### Define functions

In [ ]:
def decimal_year(dates) -> np.ndarray:
    """Timestamp(s) / date string(s) -> decimal year (e.g. 1989-03-13 -> 1989.19)."""
    d = pd.DatetimeIndex(pd.to_datetime(np.atleast_1d(dates)))
    days_in_year = np.where(d.is_leap_year, 366.0, 365.0)
    return (d.year + (d.dayofyear - 1) / days_in_year).to_numpy(dtype=float)

In [ ]:
def igrf14_dipole_unit_vector(epoch: float) -> np.ndarray:
    """
    Returns the centered-dipole axis (unit vector) in GEO Cartesian coordinates
    using IGRF-14 dipole Gauss coefficients interpolated to `epoch` (decimal year).

    NOTE: ppigrf's read_shc() index is a DatetimeIndex. It MUST be converted to
    decimal years before np.interp -- casting it to float gives nanoseconds since
    1970, which silently returns the 1970 dipole for any epoch.
    """
    g_df, h_df = core.read_shc()

    g10_series = g_df[(1, 0)].astype(float)
    g11_series = g_df[(1, 1)].astype(float)
    h11_series = h_df[(1, 1)].astype(float)

    t = decimal_year(g10_series.index)
    assert t[0] <= epoch <= t[-1], f"epoch {epoch} outside IGRF range {t[0]:.0f}-{t[-1]:.0f}"
    g10 = np.interp(epoch, t, g10_series.to_numpy())
    g11 = np.interp(epoch, t, g11_series.to_numpy())
    h11 = np.interp(epoch, t, h11_series.to_numpy())

    m = np.array([g11, h11, g10], dtype=float)
    return m / np.linalg.norm(m)

In [ ]:
def north_geomag_pole(epoch: float):
    """(lat, lon) of the IGRF north geomagnetic (dipole) pole at `epoch` -- sanity check."""
    p = -igrf14_dipole_unit_vector(epoch)
    return np.rad2deg(np.arcsin(p[2])), np.rad2deg(np.arctan2(p[1], p[0]))

# Sanity check: pole must move with epoch (published IGRF: ~78.6N 1970, ~79.1N 1989, ~80.8N 2024)
for ep in [1970.0, 1989.19, 2003.82, 2024.36]:
    print(f"epoch {ep:8.2f}: N geomag pole = (%.2f N, %.2f E)" % north_geomag_pole(ep))

In [ ]:
def geo_to_cdm(lat_deg, lon_deg, epoch: float):
    """
    Convert geographic 'geo' lat/lon to centered dipole magnetic lat/lon.
    lat_deg, lon_deg can be scalars or numpy arrays.
    Returns (cdm_lat_deg, cdm_lon_deg) with lon in [-180, 180).
    """
    lat = np.deg2rad(np.asarray(lat_deg, dtype=float))
    lon = np.deg2rad(np.asarray(lon_deg, dtype=float))

    # station unit vectors in GEO Cartesian
    r = np.column_stack([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat)
    ])

    m = igrf14_dipole_unit_vector(epoch)

    # CDM latitude
    cdm_lat = np.rad2deg(np.arcsin(r @ m))

    # define dipole coordinate basis for longitude
    z_cd = m
    y_cd = np.cross([0.0, 0.0, 1.0], z_cd)
    y_cd = y_cd / np.linalg.norm(y_cd)
    x_cd = np.cross(y_cd, z_cd)

    cdm_lon = np.rad2deg(np.arctan2(r @ y_cd, r @ x_cd))
    cdm_lon = ((cdm_lon + 180) % 360) - 180

    return cdm_lat, cdm_lon

### Run on all events

In [ ]:
%%time
results = {}
for event_id, prefix in EVENTS.items():
    epoch = float(decimal_year(event_id)[0])
    df = pd.read_csv(f'{dpath}{in_csv_fmt.format(prefix=prefix)}', encoding='utf-8-sig')
    df[LAT_COL] = df[LAT_COL].astype(float)
    df[LON_COL] = df[LON_COL].astype(float)

    cdm_lat, cdm_lon = geo_to_cdm(df[LAT_COL].to_numpy(), df[LON_COL].to_numpy(), epoch)
    df["cdm_lat"] = cdm_lat
    df["cdm_lon"] = cdm_lon
    results[event_id] = (prefix, epoch, df)
    print(f"{event_id}: epoch {epoch:.2f}, {len(df)} sites, |cdm_lat| {np.abs(cdm_lat).min():.1f}-{np.abs(cdm_lat).max():.1f} deg")

Write out to CSV (one per event):

In [ ]:
for event_id, (prefix, epoch, df) in results.items():
    out = f'{dpath}{out_csv_fmt.format(prefix=prefix)}'
    df.to_csv(out, index=False)
    print(f" ✅ Wrote: {out}")

### Map landmarks (cities shown as stars on the event maps)

Same epoch as that event's stations, so star colors are on the same scale.

In [ ]:
lm = pd.read_csv(landmarks_in)
lm_epoch = decimal_year(lm["event_id"])
lm_cdm_lat = np.empty(len(lm)); lm_cdm_lon = np.empty(len(lm))
for i, (la, lo, ep) in enumerate(zip(lm[LAT_COL], lm[LON_COL], lm_epoch)):
    a, b = geo_to_cdm(la, lo, ep)
    lm_cdm_lat[i] = np.ravel(a)[0]; lm_cdm_lon[i] = np.ravel(b)[0]
lm["epoch"] = np.round(lm_epoch, 2)
lm["cdm_lat"] = lm_cdm_lat
lm["cdm_lon"] = lm_cdm_lon
lm.to_csv(landmarks_out, index=False)
print(f" ✅ Wrote: {landmarks_out}")
lm

### Single-point lookup (city / custom coordinate)

Optional. Enter one geographic lat/lon and a decimal year to get its centered
dipole magnetic lat/lon. Does not touch any CSV output above.

In [ ]:
# --- Single-point CDM lookup (city or custom coordinate) ---
# Only needs the "Set up" and "Define functions" cells to have been run.

point_lat  = 55.6050   # deg N   (Malmö)
point_lon  = 13.0038   # deg E   (negative = west)
point_year = 2003.82   # decimal year (e.g. 1989.19, 2003.82, 2024.36)


def lookup_cdm(lat_deg, lon_deg, year, label=None):
    """Geographic lat/lon (+year) -> centered dipole magnetic lat/lon (scalars)."""
    clat, clon = geo_to_cdm(lat_deg, lon_deg, year)
    clat = float(np.ravel(clat)[0])
    clon = float(np.ravel(clon)[0])
    tag = f"{label}  " if label else ""
    print(f"{tag}geo lat/lon = ({lat_deg:.4f}, {lon_deg:.4f}), epoch {year}")
    print(f"{' ' * len(tag)}CDM lat/lon = ({clat:.2f}, {clon:.2f})")
    return clat, clon


cdm_lat_pt, cdm_lon_pt = lookup_cdm(point_lat, point_lon, point_year, label="Malmö:")